In [2]:
import pandas as pd
import numpy as np

### Loading Raw Files

In [3]:
aq = pd.read_csv("../data/Aquaponds_Dataset.csv", low_memory=False)
ponds = pd.read_csv("../data/Ponds_data.csv", low_memory=False)

In [4]:

print("Aquaponds Columns:", aq.columns.tolist())
print("Ponds Columns:", ponds.columns.tolist())

Aquaponds Columns: ['station', 'Date', 'Time', 'NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'MANGANESE(mg/l)', 'label_3class']
Ponds Columns: ['station', 'Date', 'Time', 'NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'MANGANESE(mg/l)', 'label']


In [5]:
FEATURES = [
    "NITRATE(PPM)",
    "PH",
    "AMMONIA(mg/l)",
    "TEMP",
    "DO",
    "TURBIDITY",
    "MANGANESE(mg/l)",
]

print("\n--- Aquaponds ---")
print(aq[FEATURES].dtypes)

print("\n--- Ponds_data ---")
print(ponds[FEATURES].dtypes)


--- Aquaponds ---
NITRATE(PPM)           str
PH                 float64
AMMONIA(mg/l)      float64
TEMP               float64
DO                 float64
TURBIDITY          float64
MANGANESE(mg/l)    float64
dtype: object

--- Ponds_data ---
NITRATE(PPM)           str
PH                     str
AMMONIA(mg/l)      float64
TEMP               float64
DO                     str
TURBIDITY          float64
MANGANESE(mg/l)        str
dtype: object


##### Data type Fixing

In [6]:
aq_df = aq.copy()

# converting numeric col
aq_df['NITRATE(PPM)']= pd.to_numeric(aq_df['NITRATE(PPM)'], errors='coerce')

# parsing datetime
aq_df["datetime"] = pd.to_datetime(aq_df["Date"] + " " + aq_df["Time"], dayfirst=True, errors="coerce")

aq_df.dtypes


station                       str
Date                          str
Time                          str
NITRATE(PPM)              float64
PH                        float64
AMMONIA(mg/l)             float64
TEMP                      float64
DO                        float64
TURBIDITY                 float64
MANGANESE(mg/l)           float64
label_3class                int64
datetime           datetime64[us]
dtype: object

In [7]:
pd_data = ponds.copy()
string_cols = ['NITRATE(PPM)', 'PH', 'DO', 'MANGANESE(mg/l)',]

for col in FEATURES:
    if col in pd_data.columns:
        pd_data[col] = pd.to_numeric(pd_data[col], errors='coerce')


pd_data["datetime"] = pd.to_datetime(pd_data["Date"] + " " + pd_data["Time"], dayfirst=True, errors="coerce")

pd_data.dtypes

station                       str
Date                          str
Time                          str
NITRATE(PPM)              float64
PH                        float64
AMMONIA(mg/l)             float64
TEMP                      float64
DO                        float64
TURBIDITY                 float64
MANGANESE(mg/l)           float64
label                     float64
datetime           datetime64[us]
dtype: object

In [8]:
print("Before — Aquaponds unique stations :", aq_df["station"].unique().tolist())
print("Before — Ponds_data unique stations:", pd_data["station"].unique().tolist())

aq_df["station"]      = aq_df["station"].str.lower().str.strip()
pd_data["station"] = pd_data["station"].str.lower().str.strip()

print("After  — Aquaponds  :", aq_df["station"].unique().tolist())
print("After  — Ponds_data :", pd_data["station"].unique().tolist())

Before — Aquaponds unique stations : ['Station3', 'Station2', 'station1']
Before — Ponds_data unique stations: ['station1', 'Station2', 'Station3', nan]
After  — Aquaponds  : ['station3', 'station2', 'station1']
After  — Ponds_data : ['station1', 'station2', 'station3', nan]


### Handling mising values

In [9]:
print("\n--- Aquaponds missing values ---")
aq_nulls = aq_df[FEATURES + ["datetime", "label_3class"]].isna().sum()
print(aq_nulls[aq_nulls > 0])

print("\n--- Ponds_data missing values ---")
pd_nulls = pd_data[FEATURES + ["datetime", "station", "label"]].isna().sum()
print(pd_nulls[pd_nulls > 0])


--- Aquaponds missing values ---
NITRATE(PPM)        2
TEMP                2
MANGANESE(mg/l)    10
datetime           32
dtype: int64

--- Ponds_data missing values ---
NITRATE(PPM)       44
PH                 40
AMMONIA(mg/l)      38
TEMP               44
DO                 48
TURBIDITY          38
MANGANESE(mg/l)    63
datetime           89
station            38
label              38
dtype: int64


In [10]:
print("\nDecision: DROP rows with any NaN in features/datetime/label")

aq_before = len(aq_df)
aq_df.dropna(subset=FEATURES + ["datetime", "label_3class"], inplace=True)
print(f"Aquaponds : {aq_before:,} → {len(aq_df):,} (dropped {aq_before - len(aq_df)})")

pd_before = len(pd_data)
pd_data.dropna(subset=FEATURES + ["datetime", "station", "label"], inplace=True)
print(f"Ponds_data: {pd_before:,} → {len(pd_data):,} (dropped {pd_before - len(pd_data)})")


Decision: DROP rows with any NaN in features/datetime/label
Aquaponds : 50,000 → 49,954 (dropped 46)
Ponds_data: 74,796 → 74,669 (dropped 127)


### Duplicate Removal

If the same (station, datetime) appears twice, the model
sees the same moment twice, which distorts learning.


In [11]:
aq_dups = aq_df.duplicated(subset=["station", "datetime"]).sum()
pd_dups = pd_data.duplicated(subset=["station", "datetime"]).sum()

print(f"Aquaponds  duplicate (station, datetime) pairs: {aq_dups}")
print(f"Ponds_data duplicate (station, datetime) pairs: {pd_dups}")

Aquaponds  duplicate (station, datetime) pairs: 0
Ponds_data duplicate (station, datetime) pairs: 0


### Seperating Station 2 As our main Focus

In [12]:
aq_station2 = aq_df[aq_df['station'] == 'station2'].copy()
pd_station2  = pd_data[pd_data['station'] == 'station2'].copy()


print(f"Total rows in ponds_data station2: {len(pd_station2)}")
print(f"Total rows in Acquapond station2: {len(aq_station2)}")

Total rows in ponds_data station2: 25455
Total rows in Acquapond station2: 17111


### Remaping Labels to When to Feed Classes

In [13]:
# Analysis to check what each class represent
station2_analysis = aq_station2.groupby('label_3class')\
[['DO', 'AMMONIA(mg/l)', 'TEMP']].mean()
print(station2_analysis.round(2))

                 DO  AMMONIA(mg/l)   TEMP
label_3class                             
0              5.49           0.11  27.57
1              9.45           0.04  25.39
2             16.41           0.03  24.45


AQUAPONDS label_3class is INVERTED (confirmed by data analysis):
- label 0 -> DO mean 5.49  (LOW)  + Ammonia mean 0.11 (HIGH) = STRESS
- label 1 -> DO mean 9.45  (MED)  + Ammonia mean 0.04 (MED)  = MODERATE
- label 2 -> DO mean 16.41 (HIGH) + Ammonia mean 0.03 (LOW)  = OPTIMAL

OUR FEED WINDOW CLASSES:
- 0 = Prime Feed  (feed at full ration)
- 1 = Reduce Feed (feed at 40-50%)
- 2 = Halt Feed   (stop entirely)

In [14]:
aq_station2["feed_window"] = aq_station2["label_3class"].map({0: 2, 1: 1, 2: 0})

print("Aquaponds feed_window distribution:")
for cls, name in {0: "Prime Feed", 1: "Reduce Feed", 2: "Halt Feeding"}.items():
    n = (aq_station2["feed_window"] == cls).sum()
    print(f"  Class {cls} ({name}): {n:,}")

Aquaponds feed_window distribution:
  Class 0 (Prime Feed): 5,371
  Class 1 (Reduce Feed): 6,351
  Class 2 (Halt Feeding): 5,389


In [15]:

station2_analysis2 = pd_station2.groupby('label')\
[['DO', 'AMMONIA(mg/l)', 'TEMP']].mean()
print(station2_analysis2.round(2))

          DO  AMMONIA(mg/l)   TEMP
label                             
0.0    10.69           0.06  25.87
1.0     2.21           0.05  24.26


In [16]:
pd_station2["feed_window"] = pd_station2["label"].map({0.0: 0, 1.0: 2})

print("\nPonds_data feed_window distribution:")
for cls, name in {0: "Prime Feed", 1: "Reduce Feed", 2: "Halt Feeding"}.items():
    n = (pd_station2["feed_window"] == cls).sum()
    print(f"  Class {cls} ({name}): {n:,}")


Ponds_data feed_window distribution:
  Class 0 (Prime Feed): 24,543
  Class 1 (Reduce Feed): 0
  Class 2 (Halt Feeding): 912


In [17]:
aq_station2.head()

,station,Date,Time,NITRATE(PPM),PH,AMMONIA(mg/l),TEMP,DO,TURBIDITY,MANGANESE(mg/l),label_3class,datetime,feed_window
1,station2,18-04-2022,19:00:00,40.0932,5.8882,0.126232,22.61820,6.817200,18.1620,0.64610,0,2022-04-18 19:00:00,2
5,station2,17-10-2022,18:40:00,32.4355,5.8610,0.032930,24.73110,14.998200,23.5252,0.65598,2,2022-10-17 18:40:00,0
6,station2,17-11-2022,15:20:00,17.9591,5.5068,0.027110,28.39650,5.367100,28.6691,0.48644,0,2022-11-17 15:20:00,2
13,station2,30-04-2022,12:20:00,27.5576,6.0878,0.172042,33.87164,6.933386,37.0303,0.49700,0,2022-04-30 12:20:00,2
20,station2,11-02-2022,00:20:00,41.4120,6.5650,0.045198,22.15268,4.554920,25.4448,0.76384,0,2022-02-11 00:20:00,2


In [18]:
aq_station2 = aq_station2.sort_values(["datetime"]).reset_index(drop=True)
pd_station2 = pd_station2.sort_values(["datetime"]).reset_index(drop=True)

aq_station2[["station","datetime","DO","TEMP","feed_window"]].head()

,station,datetime,DO,TEMP,feed_window
0,station2,2022-02-01 00:00:00,8.21866,22.91168,1
1,station2,2022-02-01 00:20:00,6.23826,22.77000,2
2,station2,2022-02-01 00:40:00,6.33728,22.73964,2
3,station2,2022-02-01 02:00:00,7.12944,22.09196,2
4,station2,2022-02-01 02:40:00,9.40690,21.76812,1


### Merging Datatset

In [27]:
aq_station2["source"]      = "aquaponds"
pd_station2["source"] = "ponds_data"

# Select only the columns we need going forward
KEEP_COLS = FEATURES + ["station", "datetime", "feed_window", "source"]

merged = pd.concat(
    [aq_station2[KEEP_COLS], pd_station2[KEEP_COLS]],
    ignore_index=True
)

print(merged[FEATURES].isna().sum(), "\n")
print("Total Number of duplicate", merged.duplicated(subset=["datetime"]).sum(), "\n")
print(merged[merged.duplicated(subset=['datetime'], keep=False)].sort_values(by='datetime').head())



NITRATE(PPM)       0
PH                 0
AMMONIA(mg/l)      0
TEMP               0
DO                 0
TURBIDITY          0
MANGANESE(mg/l)    0
dtype: int64 

Total Number of duplicate 17111 

       NITRATE(PPM)     PH  AMMONIA(mg/l)      TEMP       DO  TURBIDITY  \
0            29.172  5.656       0.022088  22.91168  8.21866    29.9592   
17111        29.172  5.656       0.022088  22.91168  8.21866    29.9592   
17112        14.280  5.858       0.071284  22.77000  6.23826    32.8320   
1            14.280  5.858       0.071284  22.77000  6.23826    32.8320   
2             5.712  5.454       0.058232  22.73964  6.33728    25.6500   

       MANGANESE(mg/l)   station            datetime  feed_window      source  
0              0.64480  station2 2022-02-01 00:00:00            1   aquaponds  
17111          0.64480  station2 2022-02-01 00:00:00            0  ponds_data  
17112          0.65472  station2 2022-02-01 00:20:00            0  ponds_data  
1              0.65472  station2 

In [40]:
merged.drop_duplicates(subset=['datetime'], inplace=True)

# Drop isolated temperature sensor faults (where TEMP is exactly 0)
merged = merged[merged['TEMP'] != 0]

# Drop isolated dissolved oxygen sensor faults (where DO is exactly 0)
merged = merged[merged['DO'] != 0].reset_index(drop=True)

In [46]:
# Flag suspicious values
print("\nSuspicious value flags:")
print(f"  TEMP = 0°C (sensor failure?): {(merged['TEMP'] == 0).sum()} rows")
print(f"  DO   = 0 mg/L (dead zone):    {(merged['DO'] == 0).sum()} rows")
print(f"  TEMP > 40°C (extreme heat):   {(merged['TEMP'] > 40).sum()} rows")
print(f"  DO   > 20 mg/L (supersated):  {(merged['DO'] > 20).sum()} rows")


Suspicious value flags:
  TEMP = 0°C (sensor failure?): 0 rows
  DO   = 0 mg/L (dead zone):    0 rows
  TEMP > 40°C (extreme heat):   0 rows
  DO   > 20 mg/L (supersated):  0 rows


In [42]:
print("Total rows:", merged.shape[0])

print("merge feed_window distribution:")
for cls, name in {0: "Prime Feed", 1: "Reduce Feed", 2: "Halt Feeding"}.items():
    n = (merged["feed_window"] == cls).sum()
    print(f"  Class {cls} ({name}): {n:,}")

Total rows: 25429
merge feed_window distribution:
  Class 0 (Prime Feed): 13,434
  Class 1 (Reduce Feed): 6,351
  Class 2 (Halt Feeding): 5,644


In [47]:
for col in FEATURES:
    stats = merged.groupby('feed_window')[col].describe().round(3)
    print(col)
    print(stats.to_string())

NITRATE(PPM)
               count    mean     std  min     25%     50%     75%      max
feed_window                                                               
0            13434.0  21.148  11.884  0.0  11.342  21.231  30.296  129.536
1             6351.0  19.800  11.192  0.0  10.351  19.514  29.172   41.336
2             5644.0  36.441  32.201  0.0  19.202  27.434  35.979  143.360
PH
               count   mean    std   min    25%    50%    75%    max
feed_window                                                         
0            13434.0  6.972  1.096  4.99  6.009  6.854  7.922  9.000
1             6351.0  6.700  1.180  4.99  5.689  6.451  7.729  9.000
2             5644.0  6.667  1.046  4.99  5.888  6.413  7.388  9.007
AMMONIA(mg/l)
               count   mean    std    min    25%    50%    75%    max
feed_window                                                          
0            13434.0  0.050  0.067  0.001  0.024  0.033  0.041  0.510
1             6351.0  0.044  0.032  0.00

Manganese has high overlap and zero distinct variance between classes. Combined with the domain knowledge that Ghanaian smallholders cannot easily measure Manganese with standard manual kits, this statistical profile gives us data-driven reason to drop it.

In [51]:
print(merged.columns.tolist())

columns_to_drop = ['MANGANESE(mg/l)', 'source', 'station']
final_cleaned_df = merged.drop(columns=columns_to_drop)

print(f"Remaining Columns:\n{final_cleaned_df.columns.tolist()}")
print(f"Final Shape: {final_cleaned_df.shape}")

['NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'MANGANESE(mg/l)', 'station', 'datetime', 'feed_window', 'source']
Remaining Columns:
['NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'datetime', 'feed_window']
Final Shape: (25429, 8)


### Saving Merged Data

In [53]:
final_cleaned_df.to_csv("../data/ready_for_feature_engineering.csv", index=False)